In [ ]:
#!/usr/bin/env python3
"""
Thesis supplementary analyses — run on server (JupyterHub)
Generates DotPlots, FeaturePlots, and CSV exports for Discussion threads.
All outputs saved to OUTPUT_DIR for upload to Claude.

Requires: scanpy, anndata, pandas, numpy, matplotlib
"""

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

sc.settings.figdir = './thesis_supp_analysis/'
OUTPUT_DIR = './thesis_supp_analysis/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# PATHS — adjust if needed
# ============================================================
KIDNEY_ANNOTATED = "/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/annotated-aging-soupxed.h5ad"
MUSCLE_ANNOTATED = "/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-muscle-soupxed2.h5ad"

# Condition mapping
KIDNEY_CONDITION_MAP = {
    'sg18': 'ctrl', 'sg20': 'age', 'sg24': 'DR', 'sg26': 'sAct', 'sg28': 'combi'
}
MUSCLE_CONDITION_MAP = {
    'rgzj1': 'ctrl', 'rgzj2': 'age', 'rgzj3': 'DR', 'rgzj4': 'sAct', 'rgzj5': 'combi'
}

# ============================================================
# LOAD DATA
# ============================================================
print("Loading kidney...")
k_adata = sc.read_h5ad(KIDNEY_ANNOTATED)
# Map condition names
if 'sample' in k_adata.obs.columns:
    cond_col = 'sample'
elif 'orig.ident' in k_adata.obs.columns:
    cond_col = 'orig.ident'
else:
    cond_col = k_adata.obs.columns[0]

k_adata.obs['condition'] = k_adata.obs[cond_col].map(KIDNEY_CONDITION_MAP).fillna(k_adata.obs[cond_col])

# Cell type column
if 'mixed_celltype' in k_adata.obs.columns:
    k_ct_col = 'mixed_celltype'
elif 'celltype' in k_adata.obs.columns:
    k_ct_col = 'celltype'
print(f"Kidney loaded: {k_adata.shape}, cell type col: {k_ct_col}")
print(f"Cell types: {k_adata.obs[k_ct_col].unique()}")
print(f"Conditions: {k_adata.obs['condition'].unique()}")

print("\nLoading muscle...")
m_adata = sc.read_h5ad(MUSCLE_ANNOTATED)
if 'sample' in m_adata.obs.columns:
    m_cond_col = 'sample'
elif 'orig.ident' in m_adata.obs.columns:
    m_cond_col = 'orig.ident'
else:
    m_cond_col = m_adata.obs.columns[0]

m_adata.obs['condition'] = m_adata.obs[m_cond_col].map(MUSCLE_CONDITION_MAP).fillna(m_adata.obs[m_cond_col])

if 'mixed_celltype' in m_adata.obs.columns:
    m_ct_col = 'mixed_celltype'
elif 'celltype' in m_adata.obs.columns:
    m_ct_col = 'celltype'
print(f"Muscle loaded: {m_adata.shape}, cell type col: {m_ct_col}")
print(f"Cell types: {m_adata.obs[m_ct_col].unique()}")
print(f"Conditions: {m_adata.obs['condition'].unique()}")


# ============================================================
# ANALYSIS 1: PT DAMAGE MARKERS (Thread 3)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 1: PT damage vs healthy markers")
print("="*60)

pt_damage_markers = ['Havcr1', 'Vcam1', 'Lcn2', 'Dcdc2a', 'Tpm1', 'Sox9']
pt_healthy_markers = ['Slc34a1', 'Lrp2', 'Slc5a2', 'Slc22a6', 'Slc22a8']

all_pt_markers = pt_damage_markers + pt_healthy_markers

# Check which genes exist in the data
available_pt = [g for g in all_pt_markers if g in k_adata.var_names]
missing_pt = [g for g in all_pt_markers if g not in k_adata.var_names]
print(f"Available PT markers: {available_pt}")
print(f"Missing PT markers: {missing_pt}")

# Try alternate names
alt_names = {'Dcdc2a': 'Dcdc2', 'Havcr1': 'Kim1'}
for orig, alt in alt_names.items():
    if orig in missing_pt and alt in k_adata.var_names:
        available_pt.append(alt)
        print(f"  Found alternate: {alt} for {orig}")

if available_pt:
    # DotPlot across all kidney cell types
    fig, ax = plt.subplots(figsize=(14, 8))
    sc.pl.dotplot(k_adata, var_names=available_pt, groupby=k_ct_col,
                  standard_scale='var', show=False, ax=ax)
    plt.title('PT damage & healthy markers across kidney cell types')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}01_PT_markers_all_celltypes.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 01_PT_markers_all_celltypes.png")

    # DotPlot for PT-1 and PT-2 only, split by condition
    k_pt = k_adata[k_adata.obs[k_ct_col].isin(['PT-1', 'PT-2'])].copy()
    k_pt.obs['ct_cond'] = k_pt.obs[k_ct_col].astype(str) + '_' + k_pt.obs['condition'].astype(str)

    # Order conditions
    cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']
    ct_cond_order = [f'{ct}_{c}' for ct in ['PT-1', 'PT-2'] for c in cond_order]
    ct_cond_order = [x for x in ct_cond_order if x in k_pt.obs['ct_cond'].unique()]

    sc.pl.dotplot(k_pt, var_names=available_pt, groupby='ct_cond',
                  categories_order=ct_cond_order, standard_scale='var',
                  show=False, figsize=(12, 6))
    plt.title('PT-1 vs PT-2: damage & healthy markers across conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}02_PT1_vs_PT2_by_condition.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 02_PT1_vs_PT2_by_condition.png")

    # Export mean expression per condition per cell type as CSV
    pt_expr = pd.DataFrame()
    for ct in ['PT-1', 'PT-2']:
        for cond in cond_order:
            mask = (k_adata.obs[k_ct_col] == ct) & (k_adata.obs['condition'] == cond)
            subset = k_adata[mask]
            if subset.shape[0] > 0:
                # Use raw if available
                if k_adata.raw is not None:
                    expr_data = k_adata.raw[mask]
                    avail_in_raw = [g for g in available_pt if g in expr_data.var_names]
                    means = pd.DataFrame(expr_data[:, avail_in_raw].X.toarray(),
                                         columns=avail_in_raw).mean()
                else:
                    avail_here = [g for g in available_pt if g in subset.var_names]
                    if hasattr(subset.X, 'toarray'):
                        means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                             columns=avail_here).mean()
                    else:
                        means = pd.DataFrame(subset[:, avail_here].X,
                                             columns=avail_here).mean()
                means['celltype'] = ct
                means['condition'] = cond
                means['n_cells'] = subset.shape[0]
                pt_expr = pd.concat([pt_expr, means.to_frame().T], ignore_index=True)

    pt_expr.to_csv(f'{OUTPUT_DIR}01_PT_marker_expression.csv', index=False)
    print("Saved: 01_PT_marker_expression.csv")


# ============================================================
# ANALYSIS 2: IMMUNE CELL SUBTYPING (Thread 2)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 2: Immune cell subtype markers")
print("="*60)

immune_markers = {
    'Macrophage': ['Cd68', 'Adgre1', 'Csf1r'],
    'T cell': ['Cd3d', 'Cd3e', 'Cd8a', 'Cd4'],
    'NK cell': ['Ncr1', 'Klrb1c'],
    'B cell': ['Cd19', 'Ms4a1', 'Mzb1'],
    'Dendritic': ['Cd80', 'Itgax', 'Clec9a'],
    'Monocyte': ['Cd14', 'Fcgr3'],
    'Neutrophil': ['Ly6g', 'S100a8', 'S100a9'],
    'Mast cell': ['Kit', 'Cpa3'],
    'Antigen pres.': ['H2-Aa', 'H2-Ab1', 'H2-Eb1'],
}

all_imm_markers = [g for sublist in immune_markers.values() for g in sublist]
available_imm = [g for g in all_imm_markers if g in k_adata.var_names]
missing_imm = [g for g in all_imm_markers if g not in k_adata.var_names]
print(f"Available immune markers: {len(available_imm)}/{len(all_imm_markers)}")
print(f"Missing: {missing_imm}")

# Structured marker dict with only available genes
imm_markers_avail = {}
for cat, genes in immune_markers.items():
    avail = [g for g in genes if g in k_adata.var_names]
    if avail:
        imm_markers_avail[cat] = avail

if available_imm:
    # KIDNEY — DotPlot on IMM cells split by condition
    k_imm = k_adata[k_adata.obs[k_ct_col] == 'IMM'].copy()
    sc.pl.dotplot(k_imm, var_names=imm_markers_avail, groupby='condition',
                  categories_order=['ctrl', 'age', 'DR', 'sAct', 'combi'],
                  standard_scale='var', show=False, figsize=(14, 4))
    plt.title('Kidney IMM: immune subtype markers across conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}03_kidney_IMM_subtype_markers.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 03_kidney_IMM_subtype_markers.png")

    # MUSCLE — DotPlot on Macrophages split by condition
    macro_names = ['Macrophages', 'Macrophage', 'IMM', 'Immune cells', 'Immune cells?']
    m_macro_ct = [ct for ct in macro_names if ct in m_adata.obs[m_ct_col].unique()]
    if m_macro_ct:
        m_imm = m_adata[m_adata.obs[m_ct_col].isin(m_macro_ct)].copy()
        m_imm_markers_avail = {}
        for cat, genes in immune_markers.items():
            avail = [g for g in genes if g in m_adata.var_names]
            if avail:
                m_imm_markers_avail[cat] = avail

        sc.pl.dotplot(m_imm, var_names=m_imm_markers_avail, groupby='condition',
                      categories_order=['ctrl', 'age', 'DR', 'sAct', 'combi'],
                      standard_scale='var', show=False, figsize=(14, 4))
        plt.title('Muscle macrophages/immune: subtype markers across conditions')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}04_muscle_IMM_subtype_markers.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: 04_muscle_IMM_subtype_markers.png")

    # Export mean expression CSV for kidney IMM
    imm_expr = pd.DataFrame()
    for cond in ['ctrl', 'age', 'DR', 'sAct', 'combi']:
        mask = (k_adata.obs[k_ct_col] == 'IMM') & (k_adata.obs['condition'] == cond)
        subset = k_adata[mask]
        if subset.shape[0] > 0:
            avail_here = [g for g in available_imm if g in subset.var_names]
            if hasattr(subset.X, 'toarray'):
                means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                     columns=avail_here).mean()
            else:
                means = pd.DataFrame(subset[:, avail_here].X,
                                     columns=avail_here).mean()
            means['condition'] = cond
            means['n_cells'] = subset.shape[0]
            imm_expr = pd.concat([imm_expr, means.to_frame().T], ignore_index=True)

    imm_expr.to_csv(f'{OUTPUT_DIR}02_kidney_IMM_marker_expression.csv', index=False)
    print("Saved: 02_kidney_IMM_marker_expression.csv")


# ============================================================
# ANALYSIS 3: FIBROSIS / ECM MARKERS (Thread 1 + Vermeij)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 3: Fibrosis & ECM markers")
print("="*60)

ecm_markers = ['Col1a1', 'Col1a2', 'Col3a1', 'Col4a1', 'Col4a2',
               'Col6a3', 'Fn1', 'Vim', 'Acta2', 'Tgfb1']

available_ecm = [g for g in ecm_markers if g in k_adata.var_names]
print(f"Available ECM markers: {available_ecm}")

if available_ecm:
    # Kidney — FIB + vSMC/MC + PODO across conditions
    ecm_cts = ['FIB', 'vSMC/MC', 'PODO', 'PT-1', 'PT-2', 'IMM']
    ecm_cts_avail = [ct for ct in ecm_cts if ct in k_adata.obs[k_ct_col].unique()]
    k_ecm = k_adata[k_adata.obs[k_ct_col].isin(ecm_cts_avail)].copy()
    k_ecm.obs['ct_cond'] = k_ecm.obs[k_ct_col].astype(str) + '_' + k_ecm.obs['condition'].astype(str)

    ct_cond_order = [f'{ct}_{c}' for ct in ecm_cts_avail for c in ['ctrl', 'age', 'DR', 'sAct', 'combi']]
    ct_cond_order = [x for x in ct_cond_order if x in k_ecm.obs['ct_cond'].unique()]

    sc.pl.dotplot(k_ecm, var_names=available_ecm, groupby='ct_cond',
                  categories_order=ct_cond_order, standard_scale='var',
                  show=False, figsize=(12, 10))
    plt.title('Kidney: ECM/fibrosis markers across cell types and conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}05_kidney_ECM_fibrosis_markers.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 05_kidney_ECM_fibrosis_markers.png")

    # Col3a1/Col4a1 ratio per condition in FIB
    if 'Col3a1' in available_ecm and 'Col4a1' in available_ecm:
        ratio_data = []
        for cond in ['ctrl', 'age', 'DR', 'sAct', 'combi']:
            mask = (k_adata.obs[k_ct_col] == 'FIB') & (k_adata.obs['condition'] == cond)
            subset = k_adata[mask]
            if subset.shape[0] > 0:
                if hasattr(subset.X, 'toarray'):
                    col3 = subset[:, 'Col3a1'].X.toarray().mean()
                    col4 = subset[:, 'Col4a1'].X.toarray().mean()
                else:
                    col3 = subset[:, 'Col3a1'].X.mean()
                    col4 = subset[:, 'Col4a1'].X.mean()
                ratio = col3 / col4 if col4 > 0 else np.nan
                ratio_data.append({'condition': cond, 'Col3a1_mean': col3,
                                   'Col4a1_mean': col4, 'ratio_3to4': ratio,
                                   'n_cells': subset.shape[0]})
        ratio_df = pd.DataFrame(ratio_data)
        ratio_df.to_csv(f'{OUTPUT_DIR}03_kidney_FIB_collagen_ratio.csv', index=False)
        print("Saved: 03_kidney_FIB_collagen_ratio.csv")
        print(ratio_df)


# ============================================================
# ANALYSIS 4: ENDOTHELIAL PERMEABILITY MARKERS (Thread 2)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 4: Endothelial permeability/activation markers")
print("="*60)

endo_markers = ['Ccl14', 'Cd93', 'Sele', 'Selp', 'Vcam1', 'Pecam1',
                'Cdh5', 'Icam1', 'Icam2', 'Flt1', 'Kdr', 'Tek']

available_endo = [g for g in endo_markers if g in k_adata.var_names]
print(f"Available endothelial markers: {available_endo}")

if available_endo:
    # All EC subtypes across conditions
    ec_types = [ct for ct in k_adata.obs[k_ct_col].unique() if ct.startswith('EC-')]
    k_ec = k_adata[k_adata.obs[k_ct_col].isin(ec_types)].copy()
    k_ec.obs['ct_cond'] = k_ec.obs[k_ct_col].astype(str) + '_' + k_ec.obs['condition'].astype(str)

    ct_cond_order = [f'{ct}_{c}' for ct in sorted(ec_types) for c in ['ctrl', 'age', 'DR', 'sAct', 'combi']]
    ct_cond_order = [x for x in ct_cond_order if x in k_ec.obs['ct_cond'].unique()]

    sc.pl.dotplot(k_ec, var_names=available_endo, groupby='ct_cond',
                  categories_order=ct_cond_order, standard_scale='var',
                  show=False, figsize=(12, 12))
    plt.title('Kidney ECs: permeability/activation markers across subtypes and conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}06_kidney_EC_permeability_markers.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 06_kidney_EC_permeability_markers.png")

    # Export CSV
    ec_expr = pd.DataFrame()
    for ct in sorted(ec_types):
        for cond in ['ctrl', 'age', 'DR', 'sAct', 'combi']:
            mask = (k_adata.obs[k_ct_col] == ct) & (k_adata.obs['condition'] == cond)
            subset = k_adata[mask]
            if subset.shape[0] > 0:
                avail_here = [g for g in available_endo if g in subset.var_names]
                if hasattr(subset.X, 'toarray'):
                    means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                         columns=avail_here).mean()
                else:
                    means = pd.DataFrame(subset[:, avail_here].X,
                                         columns=avail_here).mean()
                means['celltype'] = ct
                means['condition'] = cond
                means['n_cells'] = subset.shape[0]
                ec_expr = pd.concat([ec_expr, means.to_frame().T], ignore_index=True)

    ec_expr.to_csv(f'{OUTPUT_DIR}04_kidney_EC_marker_expression.csv', index=False)
    print("Saved: 04_kidney_EC_marker_expression.csv")


# ============================================================
# ANALYSIS 5: CELL TYPE COMPOSITION CHANGES (detailed)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 5: Cell type composition per condition")
print("="*60)

# Kidney
k_comp = k_adata.obs.groupby(['condition', k_ct_col]).size().unstack(fill_value=0)
k_comp_pct = k_comp.div(k_comp.sum(axis=1), axis=0) * 100
k_comp_pct = k_comp_pct.reindex(['ctrl', 'age', 'DR', 'sAct', 'combi'])
k_comp_pct.to_csv(f'{OUTPUT_DIR}05_kidney_composition_pct.csv')
print("Saved: 05_kidney_composition_pct.csv")
print(k_comp_pct[['PT-1', 'PT-2', 'IMM', 'FIB']].round(1))

# Muscle
m_comp = m_adata.obs.groupby(['condition', m_ct_col]).size().unstack(fill_value=0)
m_comp_pct = m_comp.div(m_comp.sum(axis=1), axis=0) * 100
m_comp_pct = m_comp_pct.reindex(['ctrl', 'age', 'DR', 'sAct', 'combi'])
m_comp_pct.to_csv(f'{OUTPUT_DIR}06_muscle_composition_pct.csv')
print("Saved: 06_muscle_composition_pct.csv")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("ALL OUTPUTS:")
print("="*60)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}{f}')
    print(f"  {f} ({size/1024:.1f} KB)")

print(f"\nAll files saved to: {OUTPUT_DIR}")
print("Upload the CSV files and PNG files to Claude for review.")

# Part 2

In [ ]:
#!/usr/bin/env python3
"""
Thesis supplementary analyses — PART 2
Additional receptor and MMP/TIMP markers to strengthen Discussion threads.
Run on server after Part 1.
"""

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = './thesis_supp_analysis/'
os.makedirs(OUTPUT_DIR, exist_ok=True)


KIDNEY_CONDITION_MAP = {
    'sg18': 'ctrl', 'sg20': 'age', 'sg24': 'DR', 'sg26': 'sAct', 'sg28': 'combi'
}
MUSCLE_CONDITION_MAP = {
    'rgzj1': 'ctrl', 'rgzj2': 'age', 'rgzj3': 'DR', 'rgzj4': 'sAct', 'rgzj5': 'combi'
}

# ============================================================
# LOAD DATA
# ============================================================
print("Loading kidney...")
k_adata = sc.read_h5ad(KIDNEY_ANNOTATED)
if 'sample' in k_adata.obs.columns:
    cond_col = 'sample'
elif 'orig.ident' in k_adata.obs.columns:
    cond_col = 'orig.ident'
else:
    cond_col = k_adata.obs.columns[0]
k_adata.obs['condition'] = k_adata.obs[cond_col].map(KIDNEY_CONDITION_MAP).fillna(k_adata.obs[cond_col])
k_ct_col = 'mixed_celltype' if 'mixed_celltype' in k_adata.obs.columns else 'celltype'
print(f"Kidney: {k_adata.shape}, ct col: {k_ct_col}")

print("Loading muscle...")
m_adata = sc.read_h5ad(MUSCLE_ANNOTATED)
if 'sample' in m_adata.obs.columns:
    m_cond_col = 'sample'
elif 'orig.ident' in m_adata.obs.columns:
    m_cond_col = 'orig.ident'
else:
    m_cond_col = m_adata.obs.columns[0]
m_adata.obs['condition'] = m_adata.obs[m_cond_col].map(MUSCLE_CONDITION_MAP).fillna(m_adata.obs[m_cond_col])
m_ct_col = 'mixed_celltype' if 'mixed_celltype' in m_adata.obs.columns else 'celltype'
print(f"Muscle: {m_adata.shape}, ct col: {m_ct_col}")

cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']


# ============================================================
# ANALYSIS 6: RECEPTOR GENES ON PT (Thread 3)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 6: Receptor genes on PT-1/PT-2")
print("="*60)

receptor_genes = ['Egfr', 'Tgfbr1', 'Tgfbr2', 'Pdgfra', 'Pdgfrb',
                  'Fgfr1', 'Fgfr2', 'Met', 'Igf1r', 'Insr']

available_rec = [g for g in receptor_genes if g in k_adata.var_names]
missing_rec = [g for g in receptor_genes if g not in k_adata.var_names]
print(f"Available receptors: {available_rec}")
print(f"Missing: {missing_rec}")

if available_rec:
    # PT-1 and PT-2 split by condition
    k_pt = k_adata[k_adata.obs[k_ct_col].isin(['PT-1', 'PT-2'])].copy()
    k_pt.obs['ct_cond'] = k_pt.obs[k_ct_col].astype(str) + '_' + k_pt.obs['condition'].astype(str)
    ct_cond_order = [f'{ct}_{c}' for ct in ['PT-1', 'PT-2'] for c in cond_order]
    ct_cond_order = [x for x in ct_cond_order if x in k_pt.obs['ct_cond'].unique()]

    sc.pl.dotplot(k_pt, var_names=available_rec, groupby='ct_cond',
                  categories_order=ct_cond_order, standard_scale='var',
                  show=False, figsize=(12, 6))
    plt.title('PT-1 vs PT-2: receptor expression across conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}07_PT_receptors_by_condition.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 07_PT_receptors_by_condition.png")

    # Also show across ALL kidney cell types for context
    sc.pl.dotplot(k_adata, var_names=available_rec, groupby=k_ct_col,
                  standard_scale='var', show=False, figsize=(14, 8))
    plt.title('Receptor genes across all kidney cell types')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}08_receptors_all_kidney_celltypes.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 08_receptors_all_kidney_celltypes.png")

    # Export CSV
    rec_expr = pd.DataFrame()
    for ct in ['PT-1', 'PT-2', 'FIB', 'vSMC/MC', 'PODO', 'IMM']:
        for cond in cond_order:
            mask = (k_adata.obs[k_ct_col] == ct) & (k_adata.obs['condition'] == cond)
            subset = k_adata[mask]
            if subset.shape[0] > 0:
                avail_here = [g for g in available_rec if g in subset.var_names]
                if hasattr(subset.X, 'toarray'):
                    means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                         columns=avail_here).mean()
                else:
                    means = pd.DataFrame(subset[:, avail_here].X,
                                         columns=avail_here).mean()
                means['celltype'] = ct
                means['condition'] = cond
                means['n_cells'] = subset.shape[0]
                rec_expr = pd.concat([rec_expr, means.to_frame().T], ignore_index=True)

    rec_expr.to_csv(f'{OUTPUT_DIR}07_receptor_expression.csv', index=False)
    print("Saved: 07_receptor_expression.csv")


# ============================================================
# ANALYSIS 7: MMP/TIMP MARKERS (Thread 1 — ECM remodelling)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 7: MMP/TIMP ECM remodelling markers")
print("="*60)

mmp_genes = ['Mmp2', 'Mmp3', 'Mmp9', 'Mmp14',
             'Timp1', 'Timp2', 'Timp3',
             'Ctgf', 'Serpine1', 'Lox', 'Loxl2']

available_mmp = [g for g in mmp_genes if g in k_adata.var_names]
missing_mmp = [g for g in mmp_genes if g not in k_adata.var_names]
print(f"Available MMP/TIMP: {available_mmp}")
print(f"Missing: {missing_mmp}")

# Try alternate name for Ctgf
if 'Ctgf' in missing_mmp and 'Ccn2' in k_adata.var_names:
    available_mmp.append('Ccn2')
    print("  Found alternate: Ccn2 for Ctgf")

if available_mmp:
    # FIB + vSMC/MC + PT-1 + PT-2 + IMM across conditions
    ecm_cts = ['FIB', 'vSMC/MC', 'PT-1', 'PT-2', 'IMM', 'PODO']
    ecm_cts_avail = [ct for ct in ecm_cts if ct in k_adata.obs[k_ct_col].unique()]
    k_ecm = k_adata[k_adata.obs[k_ct_col].isin(ecm_cts_avail)].copy()
    k_ecm.obs['ct_cond'] = k_ecm.obs[k_ct_col].astype(str) + '_' + k_ecm.obs['condition'].astype(str)

    ct_cond_order = [f'{ct}_{c}' for ct in ecm_cts_avail for c in cond_order]
    ct_cond_order = [x for x in ct_cond_order if x in k_ecm.obs['ct_cond'].unique()]

    sc.pl.dotplot(k_ecm, var_names=available_mmp, groupby='ct_cond',
                  categories_order=ct_cond_order, standard_scale='var',
                  show=False, figsize=(12, 12))
    plt.title('Kidney: MMP/TIMP ECM remodelling markers across cell types and conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}09_kidney_MMP_TIMP_markers.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 09_kidney_MMP_TIMP_markers.png")

    # Export MMP/TIMP mean expression for FIB across conditions
    mmp_expr = pd.DataFrame()
    for ct in ecm_cts_avail:
        for cond in cond_order:
            mask = (k_adata.obs[k_ct_col] == ct) & (k_adata.obs['condition'] == cond)
            subset = k_adata[mask]
            if subset.shape[0] > 0:
                avail_here = [g for g in available_mmp if g in subset.var_names]
                if hasattr(subset.X, 'toarray'):
                    means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                         columns=avail_here).mean()
                else:
                    means = pd.DataFrame(subset[:, avail_here].X,
                                         columns=avail_here).mean()
                means['celltype'] = ct
                means['condition'] = cond
                means['n_cells'] = subset.shape[0]
                mmp_expr = pd.concat([mmp_expr, means.to_frame().T], ignore_index=True)

    mmp_expr.to_csv(f'{OUTPUT_DIR}09_MMP_TIMP_expression.csv', index=False)
    print("Saved: 09_MMP_TIMP_expression.csv")

    # MMP2/TIMP2 ratio in FIB (enzymatic remodelling capacity)
    if 'Mmp2' in available_mmp and 'Timp2' in available_mmp:
        ratio_data = []
        for cond in cond_order:
            mask = (k_adata.obs[k_ct_col] == 'FIB') & (k_adata.obs['condition'] == cond)
            subset = k_adata[mask]
            if subset.shape[0] > 0:
                if hasattr(subset.X, 'toarray'):
                    mmp2 = subset[:, 'Mmp2'].X.toarray().mean()
                    timp2 = subset[:, 'Timp2'].X.toarray().mean()
                else:
                    mmp2 = subset[:, 'Mmp2'].X.mean()
                    timp2 = subset[:, 'Timp2'].X.mean()
                ratio = mmp2 / timp2 if timp2 > 0 else np.nan
                ratio_data.append({'condition': cond, 'Mmp2_mean': mmp2,
                                   'Timp2_mean': timp2, 'ratio_MMP2_TIMP2': ratio,
                                   'n_cells': subset.shape[0]})
        ratio_df = pd.DataFrame(ratio_data)
        ratio_df.to_csv(f'{OUTPUT_DIR}10_kidney_FIB_MMP2_TIMP2_ratio.csv', index=False)
        print("Saved: 10_kidney_FIB_MMP2_TIMP2_ratio.csv")
        print(ratio_df)


# ============================================================
# ANALYSIS 8: PDGF pathway genes (Thread 1 — colleague suggestion)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 8: PDGF pathway genes")
print("="*60)

pdgf_genes = ['Pdgfa', 'Pdgfb', 'Pdgfc', 'Pdgfd',
              'Pdgfra', 'Pdgfrb']

available_pdgf = [g for g in pdgf_genes if g in k_adata.var_names]
print(f"Available PDGF genes: {available_pdgf}")

if available_pdgf:
    # FIB + vSMC/MC + EC subtypes + PODO
    pdgf_cts = ['FIB', 'vSMC/MC', 'PODO', 'EC-1(gEC)', 'EC-2', 'EC-3', 'PT-1', 'PT-2', 'IMM']
    pdgf_cts_avail = [ct for ct in pdgf_cts if ct in k_adata.obs[k_ct_col].unique()]
    k_pdgf = k_adata[k_adata.obs[k_ct_col].isin(pdgf_cts_avail)].copy()
    k_pdgf.obs['ct_cond'] = k_pdgf.obs[k_ct_col].astype(str) + '_' + k_pdgf.obs['condition'].astype(str)

    ct_cond_order = [f'{ct}_{c}' for ct in pdgf_cts_avail for c in cond_order]
    ct_cond_order = [x for x in ct_cond_order if x in k_pdgf.obs['ct_cond'].unique()]

    sc.pl.dotplot(k_pdgf, var_names=available_pdgf, groupby='ct_cond',
                  categories_order=ct_cond_order, standard_scale='var',
                  show=False, figsize=(10, 14))
    plt.title('Kidney: PDGF ligands and receptors across cell types and conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}11_kidney_PDGF_pathway.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 11_kidney_PDGF_pathway.png")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("PART 2 — ALL NEW OUTPUTS:")
print("="*60)
new_files = [f for f in sorted(os.listdir(OUTPUT_DIR)) if f.startswith(('07', '08', '09', '10', '11'))]
for f in new_files:
    size = os.path.getsize(f'{OUTPUT_DIR}{f}')
    print(f"  {f} ({size/1024:.1f} KB)")

print(f"\nAll files saved to: {OUTPUT_DIR}")
print("Upload CSVs and PNGs to Claude for review.")

# Part 3 Muscle

In [ ]:
#!/usr/bin/env python3
"""
Thesis supplementary analyses — PART 3
Muscle-parallel analyses to match kidney findings.
Run on server after Parts 1 and 2.
"""

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = './thesis_supp_analysis/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# PATHS — same as Parts 1/2
# ============================================================
KIDNEY_ANNOTATED = "/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/annotated-aging-soupxed.h5ad"
MUSCLE_ANNOTATED = "/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-muscle-soupxed2.h5ad"

KIDNEY_CONDITION_MAP = {
    'sg18': 'ctrl', 'sg20': 'age', 'sg24': 'DR', 'sg26': 'sAct', 'sg28': 'combi'
}
MUSCLE_CONDITION_MAP = {
    'rgzj1': 'ctrl', 'rgzj2': 'age', 'rgzj3': 'DR', 'rgzj4': 'sAct', 'rgzj5': 'combi'
}

cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']

# ============================================================
# LOAD MUSCLE DATA
# ============================================================
print("Loading muscle...")
m_adata = sc.read_h5ad(MUSCLE_ANNOTATED)
if 'sample' in m_adata.obs.columns:
    m_cond_col = 'sample'
elif 'orig.ident' in m_adata.obs.columns:
    m_cond_col = 'orig.ident'
else:
    m_cond_col = m_adata.obs.columns[0]
m_adata.obs['condition'] = m_adata.obs[m_cond_col].map(MUSCLE_CONDITION_MAP).fillna(m_adata.obs[m_cond_col])
m_ct_col = 'mixed_celltype' if 'mixed_celltype' in m_adata.obs.columns else 'celltype'
print(f"Muscle: {m_adata.shape}, ct col: {m_ct_col}")
print(f"Cell types: {m_adata.obs[m_ct_col].unique()}")

# Also load kidney for cross-tissue comparison plots
print("Loading kidney...")
k_adata = sc.read_h5ad(KIDNEY_ANNOTATED)
if 'sample' in k_adata.obs.columns:
    k_cond_col = 'sample'
elif 'orig.ident' in k_adata.obs.columns:
    k_cond_col = 'orig.ident'
else:
    k_cond_col = k_adata.obs.columns[0]
k_adata.obs['condition'] = k_adata.obs[k_cond_col].map(KIDNEY_CONDITION_MAP).fillna(k_adata.obs[k_cond_col])
k_ct_col = 'mixed_celltype' if 'mixed_celltype' in k_adata.obs.columns else 'celltype'
print(f"Kidney: {k_adata.shape}, ct col: {k_ct_col}")


# ============================================================
# ANALYSIS 9: MUSCLE ECM MARKERS (parallel to kidney Analysis 3)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 9: Muscle ECM/fibrosis markers")
print("="*60)

ecm_markers = ['Col1a1', 'Col1a2', 'Col3a1', 'Col4a1', 'Col6a3',
               'Fn1', 'Vim', 'Acta2', 'Tgfb1']

available_ecm_m = [g for g in ecm_markers if g in m_adata.var_names]
missing_ecm_m = [g for g in ecm_markers if g not in m_adata.var_names]
print(f"Available ECM markers in muscle: {available_ecm_m}")
print(f"Missing: {missing_ecm_m}")

if available_ecm_m:
    # Muscle stromal cell types: FAPs, Fibroblasts, Macrophages, Endothelial cells
    m_ecm_cts = ['FAPs', 'Fibroblasts', 'Macrophages', 'Endothelial cells', 'Myonuclei']
    m_ecm_cts_avail = [ct for ct in m_ecm_cts if ct in m_adata.obs[m_ct_col].unique()]
    print(f"ECM cell types available in muscle: {m_ecm_cts_avail}")

    if m_ecm_cts_avail:
        m_ecm = m_adata[m_adata.obs[m_ct_col].isin(m_ecm_cts_avail)].copy()
        m_ecm.obs['ct_cond'] = m_ecm.obs[m_ct_col].astype(str) + '_' + m_ecm.obs['condition'].astype(str)

        ct_cond_order = [f'{ct}_{c}' for ct in m_ecm_cts_avail for c in cond_order]
        ct_cond_order = [x for x in ct_cond_order if x in m_ecm.obs['ct_cond'].unique()]

        sc.pl.dotplot(m_ecm, var_names=available_ecm_m, groupby='ct_cond',
                      categories_order=ct_cond_order, standard_scale='var',
                      show=False, figsize=(12, 10))
        plt.title('Muscle: ECM/fibrosis markers across cell types and conditions')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}12_muscle_ECM_fibrosis_markers.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: 12_muscle_ECM_fibrosis_markers.png")

        # Export CSV
        m_ecm_expr = pd.DataFrame()
        for ct in m_ecm_cts_avail:
            for cond in cond_order:
                mask = (m_adata.obs[m_ct_col] == ct) & (m_adata.obs['condition'] == cond)
                subset = m_adata[mask]
                if subset.shape[0] > 0:
                    avail_here = [g for g in available_ecm_m if g in subset.var_names]
                    if hasattr(subset.X, 'toarray'):
                        means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                             columns=avail_here).mean()
                    else:
                        means = pd.DataFrame(subset[:, avail_here].X,
                                             columns=avail_here).mean()
                    means['celltype'] = ct
                    means['condition'] = cond
                    means['n_cells'] = subset.shape[0]
                    m_ecm_expr = pd.concat([m_ecm_expr, means.to_frame().T], ignore_index=True)

        m_ecm_expr.to_csv(f'{OUTPUT_DIR}12_muscle_ECM_marker_expression.csv', index=False)
        print("Saved: 12_muscle_ECM_marker_expression.csv")

        # Collagen ratio in muscle FAPs (parallel to kidney FIB)
        fap_ct = 'FAPs' if 'FAPs' in m_ecm_cts_avail else 'Fibroblasts'
        if 'Col3a1' in available_ecm_m and 'Col4a1' in available_ecm_m:
            ratio_data = []
            for cond in cond_order:
                mask = (m_adata.obs[m_ct_col] == fap_ct) & (m_adata.obs['condition'] == cond)
                subset = m_adata[mask]
                if subset.shape[0] > 0:
                    if hasattr(subset.X, 'toarray'):
                        col3 = subset[:, 'Col3a1'].X.toarray().mean()
                        col4 = subset[:, 'Col4a1'].X.toarray().mean()
                    else:
                        col3 = subset[:, 'Col3a1'].X.mean()
                        col4 = subset[:, 'Col4a1'].X.mean()
                    ratio = col3 / col4 if col4 > 0 else np.nan
                    ratio_data.append({'condition': cond, 'Col3a1_mean': col3,
                                       'Col4a1_mean': col4, 'ratio_3to4': ratio,
                                       'n_cells': subset.shape[0], 'celltype': fap_ct})
            ratio_df = pd.DataFrame(ratio_data)
            ratio_df.to_csv(f'{OUTPUT_DIR}13_muscle_FAP_collagen_ratio.csv', index=False)
            print(f"Saved: 13_muscle_FAP_collagen_ratio.csv (in {fap_ct})")
            print(ratio_df)
        else:
            print(f"Col3a1 or Col4a1 not available in muscle — skipping ratio")


# ============================================================
# ANALYSIS 10: MUSCLE MMP/TIMP (parallel to kidney Analysis 7)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 10: Muscle MMP/TIMP markers")
print("="*60)

mmp_genes = ['Mmp2', 'Mmp3', 'Mmp9', 'Mmp14',
             'Timp1', 'Timp2', 'Timp3',
             'Serpine1', 'Lox', 'Loxl2']

available_mmp_m = [g for g in mmp_genes if g in m_adata.var_names]
missing_mmp_m = [g for g in mmp_genes if g not in m_adata.var_names]
print(f"Available MMP/TIMP in muscle: {available_mmp_m}")
print(f"Missing: {missing_mmp_m}")

# Try Ccn2 for Ctgf
if 'Ccn2' in m_adata.var_names:
    available_mmp_m.append('Ccn2')
    print("  Found Ccn2 (CTGF)")

if available_mmp_m and m_ecm_cts_avail:
    m_mmp = m_adata[m_adata.obs[m_ct_col].isin(m_ecm_cts_avail)].copy()
    m_mmp.obs['ct_cond'] = m_mmp.obs[m_ct_col].astype(str) + '_' + m_mmp.obs['condition'].astype(str)

    ct_cond_order = [f'{ct}_{c}' for ct in m_ecm_cts_avail for c in cond_order]
    ct_cond_order = [x for x in ct_cond_order if x in m_mmp.obs['ct_cond'].unique()]

    avail_mmp_plot = [g for g in available_mmp_m if g in m_mmp.var_names]
    sc.pl.dotplot(m_mmp, var_names=avail_mmp_plot, groupby='ct_cond',
                  categories_order=ct_cond_order, standard_scale='var',
                  show=False, figsize=(12, 10))
    plt.title('Muscle: MMP/TIMP ECM remodelling markers across cell types and conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}14_muscle_MMP_TIMP_markers.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 14_muscle_MMP_TIMP_markers.png")

    # MMP2/TIMP2 ratio in muscle FAPs
    fap_ct = 'FAPs' if 'FAPs' in m_ecm_cts_avail else 'Fibroblasts'
    if 'Mmp2' in available_mmp_m and 'Timp2' in available_mmp_m:
        ratio_data = []
        for cond in cond_order:
            mask = (m_adata.obs[m_ct_col] == fap_ct) & (m_adata.obs['condition'] == cond)
            subset = m_adata[mask]
            if subset.shape[0] > 0:
                if hasattr(subset.X, 'toarray'):
                    mmp2 = subset[:, 'Mmp2'].X.toarray().mean()
                    timp2 = subset[:, 'Timp2'].X.toarray().mean()
                else:
                    mmp2 = subset[:, 'Mmp2'].X.mean()
                    timp2 = subset[:, 'Timp2'].X.mean()
                ratio = mmp2 / timp2 if timp2 > 0 else np.nan
                ratio_data.append({'condition': cond, 'Mmp2_mean': mmp2,
                                   'Timp2_mean': timp2, 'ratio_MMP2_TIMP2': ratio,
                                   'n_cells': subset.shape[0], 'celltype': fap_ct})
        ratio_df = pd.DataFrame(ratio_data)
        ratio_df.to_csv(f'{OUTPUT_DIR}15_muscle_FAP_MMP2_TIMP2_ratio.csv', index=False)
        print(f"Saved: 15_muscle_FAP_MMP2_TIMP2_ratio.csv (in {fap_ct})")
        print(ratio_df)


# ============================================================
# ANALYSIS 11: MUSCLE RECEPTOR GENES (parallel to kidney Analysis 6)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 11: Muscle receptor genes")
print("="*60)

receptor_genes = ['Egfr', 'Tgfbr1', 'Tgfbr2', 'Pdgfra', 'Pdgfrb',
                  'Fgfr1', 'Fgfr2', 'Met', 'Igf1r', 'Insr']

available_rec_m = [g for g in receptor_genes if g in m_adata.var_names]
missing_rec_m = [g for g in receptor_genes if g not in m_adata.var_names]
print(f"Available receptors in muscle: {available_rec_m}")
print(f"Missing: {missing_rec_m}")

if available_rec_m:
    # All muscle cell types
    sc.pl.dotplot(m_adata, var_names=available_rec_m, groupby=m_ct_col,
                  standard_scale='var', show=False, figsize=(14, 6))
    plt.title('Receptor genes across all muscle cell types')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}16_receptors_all_muscle_celltypes.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 16_receptors_all_muscle_celltypes.png")

    # FAPs + Fibroblasts + Macrophages by condition
    rec_cts = ['FAPs', 'Fibroblasts', 'Macrophages', 'Endothelial cells']
    rec_cts_avail = [ct for ct in rec_cts if ct in m_adata.obs[m_ct_col].unique()]
    if rec_cts_avail:
        m_rec = m_adata[m_adata.obs[m_ct_col].isin(rec_cts_avail)].copy()
        m_rec.obs['ct_cond'] = m_rec.obs[m_ct_col].astype(str) + '_' + m_rec.obs['condition'].astype(str)
        ct_cond_order = [f'{ct}_{c}' for ct in rec_cts_avail for c in cond_order]
        ct_cond_order = [x for x in ct_cond_order if x in m_rec.obs['ct_cond'].unique()]

        sc.pl.dotplot(m_rec, var_names=available_rec_m, groupby='ct_cond',
                      categories_order=ct_cond_order, standard_scale='var',
                      show=False, figsize=(12, 8))
        plt.title('Muscle: receptor expression in stromal/immune cells across conditions')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}17_muscle_receptors_by_condition.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: 17_muscle_receptors_by_condition.png")

    # Export CSV
    rec_expr_m = pd.DataFrame()
    all_m_cts = list(m_adata.obs[m_ct_col].unique())
    for ct in all_m_cts:
        for cond in cond_order:
            mask = (m_adata.obs[m_ct_col] == ct) & (m_adata.obs['condition'] == cond)
            subset = m_adata[mask]
            if subset.shape[0] > 0:
                avail_here = [g for g in available_rec_m if g in subset.var_names]
                if hasattr(subset.X, 'toarray'):
                    means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                         columns=avail_here).mean()
                else:
                    means = pd.DataFrame(subset[:, avail_here].X,
                                         columns=avail_here).mean()
                means['celltype'] = ct
                means['condition'] = cond
                means['n_cells'] = subset.shape[0]
                rec_expr_m = pd.concat([rec_expr_m, means.to_frame().T], ignore_index=True)

    rec_expr_m.to_csv(f'{OUTPUT_DIR}16_muscle_receptor_expression.csv', index=False)
    print("Saved: 16_muscle_receptor_expression.csv")


# ============================================================
# ANALYSIS 12: MUSCLE ENDOTHELIAL MARKERS (parallel to kidney Analysis 4)
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 12: Muscle endothelial markers")
print("="*60)

endo_markers = ['Cd93', 'Vcam1', 'Pecam1', 'Cdh5', 'Icam1', 'Icam2',
                'Flt1', 'Kdr', 'Tek']

available_endo_m = [g for g in endo_markers if g in m_adata.var_names]
print(f"Available endothelial markers in muscle: {available_endo_m}")

if available_endo_m:
    ec_ct = 'Endothelial cells' if 'Endothelial cells' in m_adata.obs[m_ct_col].unique() else None
    if ec_ct:
        m_ec = m_adata[m_adata.obs[m_ct_col] == ec_ct].copy()

        sc.pl.dotplot(m_ec, var_names=available_endo_m, groupby='condition',
                      categories_order=cond_order, standard_scale='var',
                      show=False, figsize=(12, 4))
        plt.title('Muscle endothelial cells: permeability/activation markers across conditions')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}18_muscle_EC_markers.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: 18_muscle_EC_markers.png")

        # Export CSV
        ec_expr_m = pd.DataFrame()
        for cond in cond_order:
            mask = (m_adata.obs[m_ct_col] == ec_ct) & (m_adata.obs['condition'] == cond)
            subset = m_adata[mask]
            if subset.shape[0] > 0:
                avail_here = [g for g in available_endo_m if g in subset.var_names]
                if hasattr(subset.X, 'toarray'):
                    means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                         columns=avail_here).mean()
                else:
                    means = pd.DataFrame(subset[:, avail_here].X,
                                         columns=avail_here).mean()
                means['condition'] = cond
                means['n_cells'] = subset.shape[0]
                ec_expr_m = pd.concat([ec_expr_m, means.to_frame().T], ignore_index=True)

        ec_expr_m.to_csv(f'{OUTPUT_DIR}18_muscle_EC_marker_expression.csv', index=False)
        print("Saved: 18_muscle_EC_marker_expression.csv")


# ============================================================
# ANALYSIS 13: CROSS-TISSUE COLLAGEN RATIO COMPARISON
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 13: Cross-tissue collagen ratio comparison")
print("="*60)

# Load kidney ratio from Part 1
try:
    k_ratio = pd.read_csv(f'{OUTPUT_DIR}03_kidney_FIB_collagen_ratio.csv')
    k_ratio['tissue'] = 'Kidney (FIB)'
except:
    print("Kidney ratio file not found — run Part 1 first")
    k_ratio = None

# Load muscle ratio from Analysis 9
try:
    m_ratio = pd.read_csv(f'{OUTPUT_DIR}13_muscle_FAP_collagen_ratio.csv')
    m_ratio['tissue'] = f"Muscle ({m_ratio['celltype'].iloc[0]})"
except:
    print("Muscle ratio not yet computed")
    m_ratio = None

if k_ratio is not None and m_ratio is not None:
    # Combined comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, (ratio_df, title) in zip(axes, [
        (k_ratio, 'Kidney fibroblasts'),
        (m_ratio, f"Muscle {m_ratio['celltype'].iloc[0]}")
    ]):
        x = np.arange(len(cond_order))
        vals = ratio_df.set_index('condition').reindex(cond_order)['ratio_3to4'].astype(float).values
        ctrl_val = vals[0]
        colors = ['#0F6E56' if c == 'ctrl' else '#A32D2D' if c == 'age' else '#185FA5' for c in cond_order]

        bars = ax.bar(x, vals, color=colors, edgecolor='white', linewidth=0.8, width=0.6)
        ax.axhline(y=ctrl_val, color='#73726c', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels(['Ctrl', 'Age', 'DR', 'sAct', 'Combi'], fontsize=11, fontweight='bold')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        for i, (v, bar) in enumerate(zip(vals, bars)):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, v + 0.05, f'{v:.2f}',
                        ha='center', fontsize=9, fontweight='bold')

        # n_cells
        for i, cond in enumerate(cond_order):
            row = ratio_df[ratio_df['condition'] == cond]
            if len(row) > 0:
                n = int(float(row['n_cells'].values[0]))
                ax.text(i, -0.15, f'n={n}', ha='center', fontsize=7, color='#555555')

    axes[0].set_ylabel('Col3a1 / Col4a1 ratio', fontsize=11, fontweight='bold')

    fig.suptitle('Cross-tissue comparison: Col3a1/Col4a1 ratio in stromal cells',
                 fontsize=14, fontweight='bold', y=1.02)
    fig.text(0.5, -0.04,
             'Interstitial (Col3a1) / basement membrane (Col4a1) collagen ratio.\n'
             'Dashed line = ctrl baseline. Exploratory snRNA-seq (n=1/condition).',
             ha='center', fontsize=9, color='#555555')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}19_cross_tissue_collagen_ratio.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 19_cross_tissue_collagen_ratio.png")

    # Combined CSV
    combined = pd.concat([k_ratio, m_ratio], ignore_index=True)
    combined.to_csv(f'{OUTPUT_DIR}19_cross_tissue_collagen_ratio.csv', index=False)
    print("Saved: 19_cross_tissue_collagen_ratio.csv")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("PART 3 — ALL NEW OUTPUTS:")
print("="*60)
new_files = [f for f in sorted(os.listdir(OUTPUT_DIR)) if f.startswith(('12', '13', '14', '15', '16', '17', '18', '19'))]
for f in new_files:
    size = os.path.getsize(f'{OUTPUT_DIR}{f}')
    print(f"  {f} ({size/1024:.1f} KB)")

print(f"\nAll files saved to: {OUTPUT_DIR}")
print("Upload CSVs and PNGs to Claude for review.")

# Quick checks

In [ ]:
print('Cd24a' in k_adata.var_names, 'Itgam' in k_adata.var_names)